# LegalQA Main Stage 2/3 — chọn checkpoint và retrieval public

Notebook này nhận output Stage 1 qua `/kaggle/input`, đánh giá tất cả epoch checkpoint trên dev100 bằng METEOR, sao chép adapter tốt nhất sang một đường dẫn portable, rồi truy xuất đầy đủ cho 1000 câu public.

Trước khi chạy, Add Input output của `legalqa_main_01_qlora_train.ipynb` hoặc dataset được tạo từ output đó.

## 1. Cấu hình và tìm Stage 1 input

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys
from zipfile import ZIP_DEFLATED, ZipFile

if not Path('/kaggle').exists():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle.')

WORK_BASE = Path('/kaggle/working')
INPUT_BASE = Path('/kaggle/input')
REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
REPO_REF = 'main'
CODE = WORK_BASE / 'uit-dsc-2026-task2-legalqa'
VERSION3_ROOT = Path('/kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1')
SAVED_INDEX = VERSION3_ROOT / 'index'
SAVED_MODELS = VERSION3_ROOT / 'models'
DATASET_ROOT = Path('/kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train')
TRAIN_PATH = DATASET_ROOT / 'train.json'
TEST_PATH = DATASET_ROOT / 'public-official.json'

STAGE1_INPUT_ROOT = None  # Nếu có nhiều bản Stage 1, đặt Path('/kaggle/input/.../legalqa_main_stage1_train_v8').
RUN_ROOT = WORK_BASE / 'legalqa_main_stage2_select_retrieve_v8'
MODELS = RUN_ROOT / 'models'
DATA = RUN_ROOT / 'data_public'
INDEX = SAVED_INDEX
CFG = RUN_ROOT / 'config.json'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def resolve_input_root(configured, marker):
    if configured is not None:
        root = Path(configured)
        if not (root / marker).is_file():
            raise FileNotFoundError(f'{root / marker} không tồn tại')
        return root
    matches = sorted(INPUT_BASE.rglob(marker))
    if len(matches) != 1:
        raise RuntimeError(f'Cần đúng 1 input chứa {marker}; tìm thấy {len(matches)}: {matches}')
    return matches[0].parent

STAGE1_ROOT = resolve_input_root(STAGE1_INPUT_ROOT, 'stage1_manifest.json')
print('Stage 1 input:', STAGE1_ROOT)
print('Stage 2 output:', RUN_ROOT)

## 2. Clone code, models và kiểm tra fingerprint

In [ ]:
if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} tồn tại nhưng không phải Git repo. Hãy Restart Session.')
    subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(CODE)], check=True)

for path in [TRAIN_PATH, TEST_PATH, SAVED_INDEX / 'index_manifest.json', SAVED_INDEX / 'corpus.sqlite', SAVED_INDEX / 'dense.faiss', SAVED_MODELS / 'models.lock.json']:
    if not path.is_file():
        raise FileNotFoundError(path)
for role in ['embedding', 'reranker', 'generator']:
    if not (SAVED_MODELS / role / 'config.json').is_file():
        raise FileNotFoundError(SAVED_MODELS / role / 'config.json')

MODELS.mkdir(parents=True, exist_ok=True)
for role in ['embedding', 'reranker', 'generator']:
    link, source = MODELS / role, SAVED_MODELS / role
    if not link.exists():
        link.symlink_to(source, target_is_directory=True)
shutil.copy2(SAVED_MODELS / 'models.lock.json', MODELS / 'models.lock.json')

shared_cfg = json.loads((CODE / 'config.json').read_text(encoding='utf-8'))
if shared_cfg['evaluation'].get('primary_metric') != 'meteor' or shared_cfg['evaluation'].get('target_meteor') != 0.65:
    raise RuntimeError('Stage 2 phải chọn checkpoint bằng METEOR, target 0.65.')
CFG.write_text(json.dumps(shared_cfg, ensure_ascii=False, indent=2), encoding='utf-8')

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def run(*args):
    command = [sys.executable, '-m', 'legalqa', '--config', str(CFG), '--models', str(MODELS), *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=CODE, check=True)

commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', '--short', 'HEAD'], text=True).strip()
stage1_manifest = json.loads((STAGE1_ROOT / 'stage1_manifest.json').read_text(encoding='utf-8'))
if stage1_manifest.get('quality_version') != 'v8':
    raise RuntimeError(f'Stage 1 quality version sai: {stage1_manifest}')
if stage1_manifest.get('code_commit') != commit:
    raise RuntimeError(f'Commit lệch: Stage 1={stage1_manifest.get("code_commit")}, hiện tại={commit}')
if stage1_manifest.get('config_sha256') != file_sha256(CFG):
    raise RuntimeError('config.json khác Stage 1; không được trộn checkpoint và config.')
print('Commit:', commit)
print('Evaluation objective:', shared_cfg['evaluation'])

## 3. Môi trường và xác thực checkpoint Stage 1

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, check=True)
subprocess.run([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, check=True)
(RUN_ROOT / 'environment.freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')

SFT_DIR = STAGE1_ROOT / 'sft'
checkpoints = sorted(SFT_DIR.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
required_checkpoint = {'adapter_config.json', 'adapter_model.safetensors', 'trainer_state.json'}
if len(checkpoints) < 2:
    raise RuntimeError(f'Stage 1 thiếu epoch checkpoints: {[p.name for p in checkpoints]}')
for checkpoint in checkpoints:
    present = {p.name for p in checkpoint.iterdir() if p.is_file()}
    missing = sorted(required_checkpoint - present)
    if missing:
        raise RuntimeError(f'{checkpoint} thiếu {missing}')
if not (SFT_DIR / 'training_result.json').is_file():
    raise RuntimeError('Stage 1 thiếu training_result.json')
print('Validated checkpoints:', [p.name for p in checkpoints])

## 4. Chuẩn bị dev100 và baseline

In [ ]:
run('prepare', '--train', TRAIN_PATH, '--test', TEST_PATH, '--output', DATA)
run('audit-models')
index_manifest = json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8'))
if index_manifest.get('chunks') != 407_107 or index_manifest.get('documents') != 8_507:
    raise RuntimeError('Không phải full index Version 3.')

SELECTION_SPLIT = shared_cfg['training']['selection_split']
SELECTION_QUESTIONS = DATA / f'{SELECTION_SPLIT}.questions.json'
SELECTION_REFERENCES = DATA / f'{SELECTION_SPLIT}.references.json'
DEV_CACHE = RUN_ROOT / f'{SELECTION_SPLIT}.retrieval.json'
BASE_PRED = RUN_ROOT / f'{SELECTION_SPLIT}.base.json'
BASE_REPORT = RUN_ROOT / f'{SELECTION_SPLIT}.base.metrics.json'
run('retrieve', '--questions', SELECTION_QUESTIONS, '--index', INDEX, '--output', DEV_CACHE)
run('generate', '--questions', SELECTION_QUESTIONS, '--retrieval', DEV_CACHE, '--output', BASE_PRED)
run('evaluate', '--predictions', BASE_PRED, '--references', SELECTION_REFERENCES, '--output', BASE_REPORT, '--label', 'base_v8')

## 5. Đánh giá checkpoint và chọn theo METEOR

In [ ]:
reports = []
for checkpoint in checkpoints:
    pred_path = RUN_ROOT / f'{SELECTION_SPLIT}.{checkpoint.name}.json'
    report_path = RUN_ROOT / f'{SELECTION_SPLIT}.{checkpoint.name}.metrics.json'
    run('generate', '--questions', SELECTION_QUESTIONS, '--retrieval', DEV_CACHE, '--adapter', checkpoint, '--output', pred_path)
    run('evaluate', '--predictions', pred_path, '--references', SELECTION_REFERENCES, '--output', report_path, '--label', checkpoint.name)
    run('compare', '--baseline', BASE_REPORT, '--candidate', report_path, '--output', RUN_ROOT / f'{SELECTION_SPLIT}.{checkpoint.name}.comparison.json')
    reports.append(report_path)
run('select', '--reports', *reports, '--output', RUN_ROOT / 'selection.json')

selection_path = RUN_ROOT / 'selection.json'
selection = json.loads(selection_path.read_text(encoding='utf-8'))
source_adapter = Path(selection['prediction_manifest']['adapter_path'])
if not source_adapter.is_dir() or not (source_adapter / 'adapter_model.safetensors').is_file():
    raise RuntimeError(f'Adapter được chọn không hợp lệ: {source_adapter}')
SELECTED_ADAPTER = RUN_ROOT / 'selected_adapter'
shutil.copytree(source_adapter, SELECTED_ADAPTER, dirs_exist_ok=True)
selection['source_adapter_path'] = str(source_adapter)
selection['portable_adapter_path'] = 'selected_adapter'
selection_path.write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8')
print('Selected:', selection['label'], '| METEOR:', selection['meteor'])
print('Portable adapter:', SELECTED_ADAPTER)

## 6. Retrieval đầy đủ cho 1000 câu public

In [ ]:
PUBLIC_QUESTIONS = DATA / 'test.questions.json'
PUBLIC_CACHE = RUN_ROOT / 'public.retrieval.json'
run('retrieve', '--questions', PUBLIC_QUESTIONS, '--index', INDEX, '--output', PUBLIC_CACHE)
if not PUBLIC_CACHE.is_file():
    raise RuntimeError('Không tạo được public.retrieval.json')
print('Public retrieval MB:', round(PUBLIC_CACHE.stat().st_size / 1024**2, 2))

## 7. Manifest bàn giao cho Stage 3

In [ ]:
stage2_manifest = {
    'stage': 2,
    'quality_version': 'v8',
    'code_commit': commit,
    'config_sha256': file_sha256(CFG),
    'stage1_manifest_sha256': file_sha256(STAGE1_ROOT / 'stage1_manifest.json'),
    'selected_label': selection['label'],
    'selected_meteor': selection['meteor'],
    'selected_adapter': 'selected_adapter',
    'public_questions': 'data_public/test.questions.json',
    'public_retrieval': 'public.retrieval.json',
    'public_retrieval_sha256': file_sha256(PUBLIC_CACHE),
    'next_notebook': 'legalqa_main_03_generate_submit.ipynb',
}
(RUN_ROOT / 'stage2_manifest.json').write_text(json.dumps(stage2_manifest, ensure_ascii=False, indent=2), encoding='utf-8')

diag = RUN_ROOT / 'legalqa_main_stage2_v8_diagnostics.zip'
diag_files = [CFG, RUN_ROOT / 'stage2_manifest.json', RUN_ROOT / 'selection.json', BASE_REPORT,
              RUN_ROOT / 'environment.freeze.txt', DATA / 'data_report.json', DATA / 'split_manifest.json']
for checkpoint in checkpoints:
    diag_files += [RUN_ROOT / f'{SELECTION_SPLIT}.{checkpoint.name}.metrics.json',
                   RUN_ROOT / f'{SELECTION_SPLIT}.{checkpoint.name}.comparison.json']
with ZipFile(diag, 'w', compression=ZIP_DEFLATED) as archive:
    for path in diag_files:
        if path.is_file():
            archive.write(path, arcname=path.name)
print(json.dumps(stage2_manifest, ensure_ascii=False, indent=2))
print('Diagnostics:', diag)
print('SUCCESS Stage 2. Hãy Save output này và Add Input vào Stage 3.')